# Actividad de aprendizaje profundo (PyTorch)

En esta actividad entrenarás una red neuronal en **PyTorch** para clasificar prendas de vestir de la base de datos Fashion-MNIST (10 clases), y analizarás cómo influyen los hiperparámetros, en particular el *learning rate*, en el entrenamiento.

Para resolverla usa como guía el notebook `Script1M1U1_pytorch.ipynb` (clasificación de dígitos MNIST): allí está el preprocesamiento, el ciclo de entrenamiento, la validación y la evaluación. Las celdas marcadas con `# TODO` son las que debes completar.

## Enunciado: puntos a resolver

El script debe cumplir los siguientes criterios:

1. Construir una red neuronal densa en PyTorch, usando `nn.Sequential` (como en la guía del curso) o una clase que herede de `nn.Module`.
2. El tamaño del batch se debe fijar como el número de instancias del conjunto de entrenamiento (entrenamiento *full-batch*).
3. Debe realizarse una búsqueda de algunos hiperparámetros: número de capas, número de unidades por capa y funciones de activación, de tal forma que se maximice el accuracy. La mejor configuración se elige con el conjunto de **validación**, y su accuracy sobre el conjunto de **prueba** (test) debe ser **superior al 78 %**.
4. Luego de completar el numeral anterior, añada una nueva sección en el código. Allí se debe tomar el mejor modelo del numeral 3 y entrenarlo con diferentes valores para el parámetro *learning rate*, específicamente: 0.0001, 0.01, 0.1, 1, 10. Para cada uno de los experimentos obtenga las gráficas de la función de costo y del accuracy para los conjuntos de entrenamiento y validación, y reporte el accuracy en prueba al final (no realizar búsqueda de hiperparámetros). Con base en las gráficas concluya acerca del efecto del learning rate en el entrenamiento de una red neuronal.

**Uso correcto de los tres conjuntos** (igual que en la guía del curso):

| Conjunto | Imágenes | Para qué se usa aquí |
|---|---|---|
| Entrenamiento | 50000 | Calcular los pesos de la red. Es el batch completo. |
| Validación | 10000 | Comparar configuraciones y vigilar el entrenamiento (elegir hiperparámetros). |
| Prueba (test) | 10000 | Evaluación final del modelo elegido, sin usarla para decidir nada. |

## Etapa 1: Carga de los datos.

Antes de comenzar a definir la red neuronal, cargamos la base de datos de interés. Para esta actividad, usaremos la base de datos Fashion-MNIST, la cual consiste en identificar tipos de prendas de vestir. Documentación: https://pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html. Esta base de datos configura un problema de 10 clases.

Dependencias: `torch`, `torchvision`, `matplotlib`, `numpy`.

Se cargan las librerías necesarias

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets

# Semilla fija para que los resultados sean reproducibles
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# Se usa GPU si está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

Se carga la base de datos. Hay 60000 imágenes para el entrenamiento y 10000 en el conjunto de prueba (test). Más adelante se apartan 10000 imágenes del entrenamiento para validación.

In [2]:
fashion_train = datasets.FashionMNIST(root='./data', train=True, download=True)
fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True)

# Las imágenes se almacenan como tensores uint8 de forma (N, 28, 28)
x_train, y_train = fashion_train.data.numpy(), fashion_train.targets.numpy()
x_test, y_test = fashion_test.data.numpy(), fashion_test.targets.numpy()

In [3]:
print(x_train.shape)

(60000, 28, 28)


Las imágenes tienen tamaño 28x28. A continuación se muestran algunos ejemplos. Cada clase es un número entre 0 y 9; la lista `class_names` indica a qué prenda corresponde.

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

fig = plt.figure()
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.tight_layout()
    plt.imshow(x_train[i], cmap='gray', interpolation='none')
    plt.title("{} ({})".format(class_names[y_train[i]], y_train[i]))
    plt.xticks([])
    plt.yticks([])

## Etapa 2: Preparación de los datos

Prepara los datos como en la guía (Etapa 1: aplastar y normalizar las imágenes, y la sección "Separar un conjunto de validación"):

- Normaliza los píxeles a [0, 1] y aplana las imágenes de 28x28 a vectores de 784 valores.
- Convierte entradas a `float32` y etiquetas a `int64` (lo que espera `nn.CrossEntropyLoss`).
- Separa el entrenamiento en **50000 imágenes de entrenamiento y 10000 de validación**. El conjunto de prueba no se toca hasta el final.
- Mueve los tensores al `device`.

Define `X_train, Y_train, X_val, Y_val, X_test, Y_test`, y las constantes `N_FEATURES` (784), `N_CLASSES` (10) y `BATCH_SIZE` (igual al número de imágenes de entrenamiento).

In [ ]:
# TODO: preparar X_train, Y_train, X_val, Y_val, X_test, Y_test

## Etapa 3: Definición del modelo y búsqueda de hiperparámetros

### 3.1 Definir el modelo

Construye la red neuronal densa (guía, sección 2.1 y clase `MnistClassifier`). Recibe 784 entradas y entrega 10 salidas (logits, sin softmax).

In [ ]:
# TODO: definir la red (o la función/clase que la construye)

### 3.2 Entrenamiento y evaluación

Define lo que necesitas para entrenar y evaluar, adaptando el ciclo de la guía (secciones 2.3.1 y 2.3.2 y clase `MnistClassifier`):

- El entrenamiento es **full-batch**: cada época es un único paso de gradiente con todas las imágenes de entrenamiento (como en 2.3.1; la Etapa 5 de la guía también entrena en full-batch con la clase `MnistClassifier`).
- En cada época guarda la pérdida y el accuracy en **entrenamiento y validación** (como en 2.3.2). El conjunto de prueba no se usa aquí.
- Con batch completo se necesitan muchas épocas (cientos) para que la red aprenda. Usa SGD.
- Define también una función que mida pérdida y accuracy sobre un conjunto sin entrenar (como `evaluate` de la guía). Se usará con el conjunto de prueba solo al final.

Usa estos nombres, porque las secciones siguientes los referencian:

- `train_model(model, lr, epochs)`: entrena y devuelve un diccionario con las listas `loss`, `accuracy`, `val_loss` y `val_accuracy` (un valor por época), las mismas claves que usa `MnistClassifier.history` en la guía.
- `evaluate(model, x, y)`: devuelve `(pérdida, accuracy)`.

In [ ]:
# TODO: def train_model(model, lr, epochs): ...
# TODO: def evaluate(model, x, y): ...

### 3.3 Búsqueda de hiperparámetros

Explora varias configuraciones variando **número de capas ocultas, unidades por capa y función de activación**. Se mantiene fijo un learning rate razonable (`SEARCH_LR`) y el mismo número de épocas para que la comparación sea justa.

Requisitos:

- Recorre al menos 8 a 10 configuraciones distintas, que cubran las tres dimensiones (por ejemplo con `itertools.product` o una lista escrita a mano).
- Para cada una, crea un modelo **nuevo** (no reutilices uno ya entrenado), entrénalo con `train_model` y guarda su accuracy final en **validación** (`history['val_accuracy'][-1]`).
- Muestra una tabla ordenada con los resultados y elige la mejor configuración **según validación**.
- Al final, entrena la mejor configuración y mide **una sola vez** su accuracy en **prueba** con `evaluate`. Debe superar el **78 %**. Si no lo logra, aumenta las épocas o prueba otras arquitecturas (y vuelve a elegir con validación; no uses la prueba para decidir).

Tiempo: cada entrenamiento de 500 épocas toma desde segundos (GPU) hasta un par de minutos (CPU) según la red.

In [ ]:
SEARCH_LR = 0.1
SEARCH_EPOCHS = 500

results = []  # lista de diccionarios: configuración y accuracy de validación

# TODO: recorrer las configuraciones, entrenar y guardar en `results`, por ejemplo:
# results.append({'hidden_sizes': ..., 'activation': ..., 'val_accuracy': ...})

In [ ]:
# TODO: mostrar los resultados ordenados de mayor a menor accuracy en validación
# y guardar la mejor configuración en la variable `best_config`.

In [ ]:
# TODO: entrenar de nuevo la mejor configuración y evaluarla UNA vez en prueba:
# test_loss, test_acc = evaluate(model, X_test, Y_test)
# print('Accuracy en prueba:', test_acc)   # debe ser > 0.78

**Análisis (TODO):** escribe aquí qué configuración fue la mejor según validación, qué accuracy alcanzó en validación y en prueba, y qué tendencias observaste al variar capas, unidades y activaciones.

## Etapa 4: Efecto del learning rate

Toma la mejor configuración de la Etapa 3 y entrénala con cada uno de estos valores de learning rate: `0.0001, 0.01, 0.1, 1, 10`. **No** hagas búsqueda de hiperparámetros aquí: arquitectura y activación quedan fijas, y solo cambia el `lr`.

Para cada valor:

- Crea un modelo **nuevo** con la misma semilla (`torch.manual_seed(SEED)` antes de crearlo), para que todos partan de los mismos pesos iniciales y la única diferencia sea el learning rate.
- Entrénalo con `train_model` y las mismas épocas para todos.
- Grafica, en dos subgráficas, la **función de costo** y el **accuracy**, con las curvas de **entrenamiento y validación**.
- Al terminar, reporta en una tabla el accuracy en **prueba** de cada modelo (`evaluate(model, X_test, Y_test)`). Guarda los modelos entrenados para poder hacerlo.

Un learning rate muy alto puede hacer que el costo explote o dé `nan`; en ese caso la gráfica puede verse vacía o con escala enorme. Es parte del resultado a analizar: puedes limitar el eje y con `plt.ylim` para ver mejor las curvas.

In [ ]:
LEARNING_RATES = [0.0001, 0.01, 0.1, 1, 10]
LR_EPOCHS = 500

histories = {}  # lr -> historial devuelto por `train_model`
models = {}     # lr -> modelo entrenado

# TODO: entrenar un modelo nuevo por cada learning rate y guardar su historial y su modelo

In [ ]:
# TODO: para cada lr, una figura con dos subgráficas:
#   izquierda: costo en entrenamiento y validación
#   derecha: accuracy en entrenamiento y validación
# (history['loss'], history['val_loss'], history['accuracy'], history['val_accuracy'])

In [ ]:
# TODO: tabla con el accuracy en prueba de cada learning rate, usando evaluate(models[lr], X_test, Y_test)

### Conclusión

Con base en las gráficas, concluye acerca del efecto del learning rate en el entrenamiento de una red neuronal.